# Dataset Inspection
Inspect `C:/dataset` (borehole `.npy` files) and `C:/dataset_raw_maps` (HDF5 map cache).

In [1]:
import numpy as np
import h5py
import pandas as pd
import pickle

## C:/dataset — config.pkl

In [2]:
with open('C:/dataset/config.pkl', 'rb') as f:
    config = pickle.load(f)

pd.DataFrame([{'key': k, 'value': str(v)} for k, v in config.items()])

,key,value
0,variables,"['rhob', 'gr_api', 'dt_us_ft', 'nphi', 'res_de..."
1,n_maps,10000
2,seed,42


## C:/dataset — stats.pkl

In [3]:
with open('C:/dataset/stats.pkl', 'rb') as f:
    stats = pickle.load(f)

pd.DataFrame(
    [{'variable': var, 'mean': round(mu, 6), 'std': round(sd, 6)} for var, (mu, sd) in stats.items()]
)

,variable,mean,std
0,rhob,2.339484,0.267300
1,gr_api,55.071285,32.209785
2,dt_us_ft,93.348991,34.641941
3,nphi,0.231165,0.144702
4,res_deep_log,0.716822,0.762032


## C:/dataset — boreholes_00000.npy
Variable indices correspond to the `variables` list in config.pkl.

In [4]:
bh = np.load('C:/dataset/boreholes_00000.npy').astype(np.float32)
variables = config.get('variables', [f'var_{i}' for i in range(bh.shape[1])])

rows = []
for i, var in enumerate(variables):
    v = bh[:, i, :]
    rows.append({
        'variable': var,
        'array shape': str(v.shape),
        'dtype': str(bh.dtype),
        'min': round(float(v.min()), 4),
        'max': round(float(v.max()), 4),
        'mean': round(float(v.mean()), 4),
        'std': round(float(v.std()), 4),
        'n_nans': int(np.isnan(v).sum()),
    })

print(f'Full array shape: {bh.shape}  (n_boreholes, n_variables, n_depth)')
pd.DataFrame(rows)

Full array shape: (1024, 5, 440)  (n_boreholes, n_variables, n_depth)


,variable,array shape,dtype,min,max,mean,std,n_nans
0,rhob,"(1024, 440)",float32,-2.8008,2.3984,0.2343,0.8571,0
1,gr_api,"(1024, 440)",float32,-1.5625,3.5801,0.0533,0.9781,0
2,dt_us_ft,"(1024, 440)",float32,-1.2930,2.9590,-0.1222,0.8279,0
3,nphi,"(1024, 440)",float32,-1.8135,2.2441,-0.0681,0.6990,0
4,res_deep_log,"(1024, 440)",float32,-1.6562,3.3965,-0.2891,0.5793,0


## C:/dataset — labels_00000.npz

In [5]:
labels = np.load('C:/dataset/labels_00000.npz')

rows = []
for key in labels.files:
    arr = labels[key]
    rows.append({
        'key': key,
        'shape': str(arr.shape),
        'dtype': str(arr.dtype),
        'min': int(arr.min()),
        'max': int(arr.max()),
        'unique values': int(np.unique(arr).size),
    })

pd.DataFrame(rows)

,key,shape,dtype,min,max,unique values
0,rocks,"(1024, 440)",int8,0,17,8
1,formations,"(1024, 440)",int8,0,12,7


## C:/dataset_raw_maps — HDF5 file-level metadata

In [6]:
with h5py.File('C:/dataset_raw_maps/raw_pool.h5', 'r') as f:
    attrs = dict(f.attrs)
    n_maps = attrs.get('n_maps', '?')
    top_keys = list(f.keys())

print(f'Number of maps : {n_maps}')
print(f'Top-level keys : {top_keys}')
pd.DataFrame([{'attr': k, 'value': str(v)} for k, v in attrs.items()])

Number of maps : 100
Top-level keys : ['maps']


,attr,value
0,belief_cfg,"{""n_maps"": 100, ""samples_per_map"": 20, ""min_dr..."
1,cache_version,1
2,n_maps,100
3,sim_cfg,"{""n_x"": 32, ""n_y"": 32, ""n_depth"": 440, ""max_de..."


## C:/dataset_raw_maps — First map: true_map datasets

In [7]:
with h5py.File('C:/dataset_raw_maps/raw_pool.h5', 'r') as f:
    first_key = sorted(f['maps'].keys())[0]
    grp = f['maps'][first_key]
    tm = grp['true_map']

    rows = []
    for name in tm.keys():
        ds = tm[name]
        if hasattr(ds, 'shape'):  # dataset, not group
            rows.append({'key': f'true_map/{name}', 'shape': str(ds.shape), 'dtype': str(ds.dtype)})
        else:  # group (variables)
            for sub_name, sub_ds in ds.items():
                rows.append({'key': f'true_map/{name}/{sub_name}', 'shape': str(sub_ds.shape), 'dtype': str(sub_ds.dtype)})

print(f'Map key: {first_key}')
pd.DataFrame(rows)

Map key: 000000


,key,shape,dtype
0,true_map/depth_axis,"(440,)",float32
1,true_map/formations,"(32, 32, 440)",object
2,true_map/rock_types,"(32, 32, 440)",object
3,true_map/variables/dt_us_ft,"(32, 32, 440)",float32
4,true_map/variables/gr_api,"(32, 32, 440)",float32
5,true_map/variables/nphi,"(32, 32, 440)",float32
6,true_map/variables/res_deep_log,"(32, 32, 440)",float32
7,true_map/variables/rhob,"(32, 32, 440)",float32
8,true_map/yield_field,"(32, 32, 440)",float32


## C:/dataset_raw_maps — First map: drill patterns & target

In [8]:
with h5py.File('C:/dataset_raw_maps/raw_pool.h5', 'r') as f:
    first_key = sorted(f['maps'].keys())[0]
    grp = f['maps'][first_key]

    rows = []
    for name in ('drill_locs', 'drill_ore_vals', 'drill_counts', 'target'):
        ds = grp[name]
        arr = ds[:]
        rows.append({
            'key': name,
            'shape': str(arr.shape),
            'dtype': str(arr.dtype),
            'min': round(float(arr.min()), 4),
            'max': round(float(arr.max()), 4),
        })

pd.DataFrame(rows)

,key,shape,dtype,min,max
0,drill_locs,"(20, 15, 2)",int16,0.0,31.0
1,drill_ore_vals,"(20, 15)",float32,0.0,0.0
2,drill_counts,"(20,)",int16,1.0,15.0
3,target,"(32, 32)",float32,0.0,0.0
